In [12]:
import numpy as np
from collections import defaultdict
with open("corpus.txt", "r") as file:
    corpus = file.read().splitlines()
corpus = [word.lower().strip() for word in corpus if word.strip()]
print("=" * 50)
print("CORPUS")
print("=" * 50)
for word in corpus:
    print(word)
vocab = {}
for word in corpus:
    chars = " ".join(list(word)) + " </w>"
    if chars not in vocab:
        vocab[chars] = 1
    else:
        vocab[chars] += 1
print("\n" + "=" * 50)
print("INITIAL VOCABULARY")
print("=" * 50)
for word, freq in vocab.items():
    print(f"{word} : {freq}")
frequencies = np.array(list(vocab.values()))
print("\nVocabulary Size :", len(vocab))
print("Total Words     :", np.sum(frequencies))
print("Maximum Frequency :", np.max(frequencies))
def get_pair_counts(vocabulary):
    pairs = defaultdict(int)
    for word, freq in vocabulary.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pairs[pair] += freq
    return pairs
def show_pair_counts(pair_counts):
    print("\n" + "=" * 50)
    print("PAIR FREQUENCIES")
    print("=" * 50)
    for pair, freq in sorted(pair_counts.items(),key=lambda x: x[1],reverse=True):
        print(f"{pair} --> {freq}")
pair_counts = get_pair_counts(vocab)
show_pair_counts(pair_counts)
def get_best_pair(pair_counts):
    if len(pair_counts) == 0:
        return None
    best_pair = max(pair_counts, key=pair_counts.get)
    return best_pair
best = get_best_pair(pair_counts)
print("\nMost Frequent Pair :", best)
print("Frequency :", pair_counts[best])
def merge_pair(pair, vocabulary):
    new_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word in vocabulary:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocabulary[word]
    return new_vocab
merge_rules = []
num_merges = 10
print("\n" + "=" * 50)
print("BPE TRAINING")
print("=" * 50)
for iteration in range(num_merges):
    pair_counts = get_pair_counts(vocab)
    if len(pair_counts) == 0:
        break
    best_pair = get_best_pair(pair_counts)
    merge_rules.append(best_pair)
    print("\n-------------------------------------")
    print("Merge :", iteration + 1)
    print("Best Pair :", best_pair)
    print("Frequency :", pair_counts[best_pair])
    vocab = merge_pair(best_pair, vocab)
    print("\nUpdated Vocabulary:\n")
    for word, freq in vocab.items():
        print(f"{word} : {freq}")
final_vocab = set()
for word in vocab:
    symbols = word.split()
    for symbol in symbols:
        final_vocab.add(symbol)
print("\n" + "=" * 50)
print("FINAL VOCABULARY")
print("=" * 50)
for token in sorted(final_vocab):
    print(token)
print("\nVocabulary Size :", len(final_vocab))
print("\n" + "=" * 50)
print("LEARNED MERGE RULES")
print("=" * 50)
for index, rule in enumerate(merge_rules, start=1):
    print(f"{index}. {rule[0]} + {rule[1]} --> {rule[0] + rule[1]}")
print("\n" + "=" * 50)
print("FINAL WORD REPRESENTATIONS")
print("=" * 50)
for word, freq in vocab.items():
    print(f"{word} : {freq}")
print("\n" + "=" * 50)
print("TRAINING SUMMARY")
print("=" * 50)
print("Number of Merge Operations :", len(merge_rules))
print("Final Vocabulary Size      :", len(final_vocab))
print("Total Words Processed      :", np.sum(frequencies))
print("\nTraining Completed Successfully!")
def encode(word, merge_rules):
    tokens = list(word) + ["</w>"]
    for pair in merge_rules:
        merged = "".join(pair)
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                tokens = (
                    tokens[:i]
                    + [merged]
                    + tokens[i + 2:]
                )
            else:
                i += 1
    return tokens
def decode(tokens):
    word = ""
    for token in tokens:
        if token != "</w>":
            word += token
    return word
test_words = [
    "lowest",
    "lower",
    "newest",
    "widest"
]
print("\n" + "=" * 50)
print("TOKENIZER TEST")
print("=" * 50)
for word in test_words:
    encoded = encode(word, merge_rules)
    decoded = decode(encoded)
    print("\nOriginal Word :", word)
    print("Encoded Tokens:", encoded)
    print("Decoded Word  :", decoded)
print("\n" + "=" * 50)
print("PROGRAM COMPLETED SUCCESSFULLY")
print("=" * 50)

CORPUS
low
lower
lowest
new
newer
widest

INITIAL VOCABULARY
l o w </w> : 1
l o w e r </w> : 1
l o w e s t </w> : 1
n e w </w> : 1
n e w e r </w> : 1
w i d e s t </w> : 1

Vocabulary Size : 6
Total Words     : 6
Maximum Frequency : 1

PAIR FREQUENCIES
('l', 'o') --> 3
('o', 'w') --> 3
('w', 'e') --> 3
('w', '</w>') --> 2
('e', 'r') --> 2
('r', '</w>') --> 2
('e', 's') --> 2
('s', 't') --> 2
('t', '</w>') --> 2
('n', 'e') --> 2
('e', 'w') --> 2
('w', 'i') --> 1
('i', 'd') --> 1
('d', 'e') --> 1

Most Frequent Pair : ('l', 'o')
Frequency : 3

BPE TRAINING

-------------------------------------
Merge : 1
Best Pair : ('l', 'o')
Frequency : 3

Updated Vocabulary:

lo w </w> : 1
lo w e r </w> : 1
lo w e s t </w> : 1
n e w </w> : 1
n e w e r </w> : 1
w i d e s t </w> : 1

-------------------------------------
Merge : 2
Best Pair : ('lo', 'w')
Frequency : 3

Updated Vocabulary:

low </w> : 1
low e r </w> : 1
low e s t </w> : 1
n e w </w> : 1
n e w e r </w> : 1
w i d e s t </w> : 1

-----------